In [ ]:
import json
from pathlib import Path

import h5py
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

# ============================================================
# 0) 配置：数据路径（硬编码）+ 输出到 ipynb 同目录
# ============================================================
DATA_DIR = Path("/home/jovyan/gpu_space/workspace_jiayi/alex_datasets/3D_minimal")
OUT_DIR  = Path(".")  # ipynb 同目录
REFERENCE_MODALITY = 341  # 0-based: MPRAGE（与你脚本一致）

AXIS_NAME = "axial"   # 只看一个轴位：axial/coronal/sagittal
AXIS_MAP = {"sagittal": 0, "coronal": 1, "axial": 2}
AXIS = AXIS_MAP[AXIS_NAME]

MIN_BRAIN_RATIO = 0.05
MAX_PIXELS = 50_000
RANDOM_SEED = 42

# “坏例子”扰动：对不同模态家族做不同平移（像素）
# 只用于示意图，不作为真实 QC 结论
SHIFT_RULES = [
    ("QTI",      range(0, 15),      (+6,  0)),
    ("b-tensor", range(15, 225),    (-4, +4)),
    ("CESTpar",  range(225, 229),   (+0, -6)),
    ("Z+M0",     range(229, 341),   (+4, +2)),
    ("QSM/SMWI", range(342, 351),   (-6, -2)),
    # MPRAGE(341) 不动
]

# 用于图右侧的“少量家族标签”（避免左侧重叠）
# [修改] 只保留前 341 个维度，去掉了 MPRAGE 和 QSM
FAMILY_LABELS = [
    ("QTI (1–15)",             0,  14),
    ("b-tensor (16–225)",     15, 224),
    ("CEST param (226–229)", 225, 228),
    ("Z-spectrum+M0 (230–341)",229, 340),
]

# ============================================================
# 1) Thesis 风格（嵌入字体、干净、适合 LaTeX）
# ============================================================
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Arial", "Liberation Sans"],
    "font.size": 9,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 3.0,
    "ytick.major.size": 3.0,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "axes.unicode_minus": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
})

# ============================================================
# 2) 读取：只读一个 slice（避免把 384×336×256×351 全载入内存）
# ============================================================
def choose_slice_max_mask(mask3d: np.ndarray, axis: int, min_ratio: float) -> tuple[int, float]:
    # mask3d shape: (X,Y,Z)
    if axis == 0:   # sagittal -> vary X
        ratios = mask3d.mean(axis=(1, 2))
    elif axis == 1: # coronal  -> vary Y
        ratios = mask3d.mean(axis=(0, 2))
    elif axis == 2: # axial    -> vary Z
        ratios = mask3d.mean(axis=(0, 1))
    else:
        raise ValueError("axis must be 0/1/2")

    valid = np.where(ratios >= min_ratio)[0]
    if valid.size == 0:
        k = int(np.argmax(ratios))
        return k, float(ratios[k])
    k = int(valid[np.argmax(ratios[valid])])
    return k, float(ratios[k])

def read_mask_slice(f: h5py.File, axis: int, slice_idx: int) -> np.ndarray:
    m = f["region_mask"]
    if axis == 0: return np.asarray(m[slice_idx, :, :]).astype(bool)
    if axis == 1: return np.asarray(m[:, slice_idx, :]).astype(bool)
    return np.asarray(m[:, :, slice_idx]).astype(bool)

def read_slice_hwc_351(f: h5py.File, axis: int, slice_idx: int) -> np.ndarray:
    """
    Return slice (H,W,351), supports:
      - stored (351,X,Y,Z)
      - stored (X,Y,Z,351)
    """
    d = f["data"]
    shp = d.shape
    if len(shp) != 4:
        raise ValueError(f"Unexpected data shape: {shp}")

    if shp[0] == 351:
        # (C,X,Y,Z)
        if axis == 0:
            arr = np.asarray(d[:, slice_idx, :, :])      # (C,Y,Z)
        elif axis == 1:
            arr = np.asarray(d[:, :, slice_idx, :])      # (C,X,Z)
        else:
            arr = np.asarray(d[:, :, :, slice_idx])      # (C,X,Y)
        arr = np.moveaxis(arr, 0, -1)                    # (H,W,C)
    elif shp[-1] == 351:
        # (X,Y,Z,C)
        if axis == 0:
            arr = np.asarray(d[slice_idx, :, :, :])      # (Y,Z,C)
        elif axis == 1:
            arr = np.asarray(d[:, slice_idx, :, :])      # (X,Z,C)
        else:
            arr = np.asarray(d[:, :, slice_idx, :])      # (X,Y,C)
    else:
        raise ValueError(f"Cannot infer channel axis from shape: {shp}")

    if arr.shape[-1] != 351:
        raise ValueError(f"Expected last dim 351, got {arr.shape}")
    return arr.astype(np.float32, copy=False)

# ============================================================
# 3) “坏例子”生成：对模态家族做平移（不 wrap，空出来补 0）
# ============================================================
def shift_hwk(img_hwk: np.ndarray, dx: int, dy: int) -> np.ndarray:
    """Shift in-plane by (dx,dy). dx->columns, dy->rows. Zeros padded."""
    H, W, K = img_hwk.shape
    out = np.zeros_like(img_hwk)
    xs_src = slice(max(0, -dx), min(W, W - dx))
    xs_dst = slice(max(0,  dx), min(W, W + dx))
    ys_src = slice(max(0, -dy), min(H, H - dy))
    ys_dst = slice(max(0,  dy), min(H, H + dy))
    out[ys_dst, xs_dst, :] = img_hwk[ys_src, xs_src, :]
    return out

def simulate_misregistration(slice_hwc: np.ndarray) -> np.ndarray:
    bad = slice_hwc.copy()
    for _, ch_range, (dx, dy) in SHIFT_RULES:
        idx = list(ch_range)
        if REFERENCE_MODALITY in idx:
            idx.remove(REFERENCE_MODALITY)
        if not idx:
            continue
        src = slice_hwc[:, :, idx]
        bad[:, :, idx] = shift_hwk(src, dx=dx, dy=dy)
    return bad

# ============================================================
# 4) Curtain（同一像素列，Good vs Bad 共用同一归一化）
# ============================================================
def build_curtains_good_bad(slice_good: np.ndarray, slice_bad: np.ndarray, mask2d: np.ndarray,
                            max_pixels: int, seed: int):
    H, W, C = slice_good.shape
    assert C == 351 and slice_bad.shape == slice_good.shape

    idx_all = np.flatnonzero(mask2d.ravel())
    if idx_all.size == 0:
        raise ValueError("Mask slice empty.")

    rng = np.random.default_rng(seed)
    if idx_all.size > max_pixels:
        idx = rng.choice(idx_all, size=max_pixels, replace=False)
        idx.sort()  # 保持“flattened spatial index”的顺序感
    else:
        idx = idx_all

    Vg = slice_good.reshape(-1, C)[idx, :]   # (N,351)
    Vb = slice_bad.reshape(-1, C)[idx, :]

    # 用 Good 的 per-modality (1,99) 分位做统一标定（两幅可比）
    p1 = np.percentile(Vg, 1, axis=0)
    p99 = np.percentile(Vg, 99, axis=0)
    denom = (p99 - p1) + 1e-8

    Ng = np.clip((Vg - p1) / denom, 0.0, 1.0)
    Nb = np.clip((Vb - p1) / denom, 0.0, 1.0)

    curtain_good = (Ng * 255).astype(np.uint8).T  # (351,N)
    curtain_bad  = (Nb * 255).astype(np.uint8).T

    meta = {
        "n_pixels_total": int(idx_all.size),
        "n_pixels_used": int(idx.size),
        "axis": AXIS_NAME,
        "max_pixels": int(max_pixels),
        "seed": int(seed),
        "norm": "per-modality percentile (1,99) computed on GOOD slice pixels; applied to both panels",
        "shift_rules": [{ "family": name, "dx": dx, "dy": dy, "n_channels": len(list(rg)) }
                        for name, rg, (dx, dy) in SHIFT_RULES],
    }
    return curtain_good, curtain_bad, meta

# ============================================================
# 5) 画图（上下两幅），右侧少量家族标签，保存到 ipynb 同目录
# ============================================================
def add_family_labels_right(ax, n_rows: int):
    for label, a, b in FAMILY_LABELS:
        mid = 0.5 * (a + b)
        y_frac = 1.0 - (mid + 0.5) / n_rows
        ax.text(1.01, y_frac, label, transform=ax.transAxes,
                va="center", ha="left", fontsize=7, clip_on=False)

def draw_family_boundaries(ax):
    for _, _, b in FAMILY_LABELS:
        ax.axhline(b + 0.5, lw=0.4, color="0.75")

# ============================================================
# 6) 主流程：默认第一个 patient
# ============================================================
mat_files = sorted(DATA_DIR.glob("*.mat"))
if not mat_files:
    raise FileNotFoundError(f"No .mat files found in {DATA_DIR}")
mat_path = mat_files[0]
subject_id = mat_path.stem
print(f"[INFO] Using subject: {subject_id}")
print(f"[INFO] Source: {mat_path}")

with h5py.File(mat_path, "r") as f:
    mask3d = np.asarray(f["region_mask"][:]).astype(bool)

slice_idx, mask_ratio = choose_slice_max_mask(mask3d, axis=AXIS, min_ratio=MIN_BRAIN_RATIO)
print(f"[INFO] Selected {AXIS_NAME} slice={slice_idx} (mask_ratio={mask_ratio:.3f})")

with h5py.File(mat_path, "r") as f:
    mask2d = read_mask_slice(f, axis=AXIS, slice_idx=slice_idx)
    slice_good = read_slice_hwc_351(f, axis=AXIS, slice_idx=slice_idx)

slice_bad = simulate_misregistration(slice_good)

curt_good, curt_bad, meta = build_curtains_good_bad(
    slice_good=slice_good,
    slice_bad=slice_bad,
    mask2d=mask2d,
    max_pixels=MAX_PIXELS,
    seed=RANDOM_SEED,
)

# ============================================================
# [修改] 数据裁剪：只保留前 341 个维度，避免 MPRAGE/QSM 标签重叠
# ============================================================
CUTOFF_DIM = 341
curt_good = curt_good[:CUTOFF_DIM, :]
curt_bad  = curt_bad[:CUTOFF_DIM, :]

# ---- Plot
fig, axs = plt.subplots(2, 1, figsize=(7.0, 6.2), sharex=True)

for ax, img, title in [
    (axs[0], curt_good, "Curtain plot (Good) — original subject slice"),
    (axs[1], curt_bad,  "Curtain plot (Bad, simulated) — modality-family in-plane shifts"),
]:
    ax.imshow(img, cmap="gray", aspect="auto", interpolation="nearest", rasterized=True)
    ax.set_yticks([])      # 避免左侧文字重叠
    ax.set_xticks([])
    draw_family_boundaries(ax)
    add_family_labels_right(ax, n_rows=img.shape[0])
    ax.set_title(title, fontsize=9)

axs[1].set_xlabel(f"Flattened spatial index within ROI (sampled N={meta['n_pixels_used']:,})")
axs[0].set_ylabel("")
axs[1].set_ylabel("")

fig.tight_layout()

base = OUT_DIR / f"fig4_1_curtain_good_vs_bad_{subject_id}_{AXIS_NAME}_slice{slice_idx:03d}"
fig.savefig(base.with_suffix(".pdf"), bbox_inches="tight", pad_inches=0.02)
fig.savefig(base.with_suffix(".png"), dpi=600, bbox_inches="tight", pad_inches=0.02)
plt.show()

# ---- Meta（可复现性）
meta_out = {
    "subject": subject_id,
    "mat_path": str(mat_path),
    "axis": AXIS_NAME,
    "slice_idx": int(slice_idx),
    "mask_ratio": float(mask_ratio),
    "displayed_dims": f"1-{CUTOFF_DIM}",
    "outputs": {"pdf": str(base.with_suffix(".pdf")), "png": str(base.with_suffix(".png"))},
    **meta
}
with open(base.with_suffix(".meta.json"), "w", encoding="utf-8") as f:
    json.dump(meta_out, f, indent=2, ensure_ascii=False)

print(f"[OK] Saved: {base.with_suffix('.pdf')}")
print(f"[OK] Saved: {base.with_suffix('.png')}")
print(f"[OK] Meta : {base.with_suffix('.meta.json')}")